# Referent × Valence Interaction Analysis

Standalone follow-up analysis using previously saved hidden-state activations.

Question:

> Does the positive-to-negative representational shift depend on who the outcome concerns?

Referent = `SELF`, `HUMAN`, or `OTHER_AI`.

For each semantic domain:

```text
V_self     = SELF_positive     - SELF_negative
V_human    = HUMAN_positive    - HUMAN_negative
V_other_ai = OTHER_AI_positive - OTHER_AI_negative
```

We compare the magnitude of these valence shifts. A larger SELF effect means positive-vs-negative outcomes produce a larger representational displacement when the outcome concerns the model itself.

This is exploratory and should not be interpreted as evidence of subjective affect.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path("representation_results")
FIGURE_DIR = Path("figures/referent_valence")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(RESULTS_DIR / "activation_manifest.csv")
manifest["path"] = manifest["path"].apply(lambda p: str(RESULTS_DIR / Path(p).name))

def load_vec(path):
    return np.load(path)

baseline_manifest = manifest[manifest["label"].isin(["positive", "negative"])].copy()
example = load_vec(baseline_manifest.iloc[0]["path"])
layers = np.arange(1, example.shape[0] + 1)

print("Rows:", len(baseline_manifest))
print("Activation shape:", example.shape)
print("Domains:", sorted(baseline_manifest["domain"].unique()))


In [ ]:
def mean_activation(domain, referent, label):
    rows = baseline_manifest[
        (baseline_manifest["domain"] == domain)
        & (baseline_manifest["referent"] == referent)
        & (baseline_manifest["label"] == label)
    ]
    if rows.empty:
        raise ValueError(f"Missing: {domain}, {referent}, {label}")
    return np.stack([load_vec(p) for p in rows["path"]]).mean(axis=0)

domains = sorted(baseline_manifest["domain"].unique())
records = []

for domain in domains:
    v_self = mean_activation(domain, "self", "positive") - mean_activation(domain, "self", "negative")
    v_human = mean_activation(domain, "human", "positive") - mean_activation(domain, "human", "negative")
    v_other = mean_activation(domain, "other_ai", "positive") - mean_activation(domain, "other_ai", "negative")

    self_norm = np.linalg.norm(v_self, axis=1)
    human_norm = np.linalg.norm(v_human, axis=1)
    other_norm = np.linalg.norm(v_other, axis=1)

    for i, layer in enumerate(layers):
        records.append({
            "domain": domain,
            "layer": layer,
            "self_norm": self_norm[i],
            "human_norm": human_norm[i],
            "other_ai_norm": other_norm[i],
            "self_minus_human": self_norm[i] - human_norm[i],
            "self_minus_other_ai": self_norm[i] - other_norm[i],
        })

interaction_df = pd.DataFrame(records)
interaction_df.head()


## Mean referent × valence interaction

`0` means SELF and the comparison referent show equal-magnitude positive-to-negative shifts.

Positive values mean the valence shift is stronger for SELF.


In [ ]:
summary = interaction_df.groupby("layer")[["self_minus_human","self_minus_other_ai"]].mean()

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(summary.index, summary["self_minus_human"], label="SELF effect − HUMAN effect")
ax.plot(summary.index, summary["self_minus_other_ai"], label="SELF effect − OTHER_AI effect")
ax.axhline(0, linewidth=1)
ax.set_xlabel("Transformer layer")
ax.set_ylabel("Extra SELF valence magnitude")
ax.set_title("Referent × valence interaction")
ax.legend()
fig.tight_layout()

path = FIGURE_DIR / "referent_x_valence_interaction.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
print("saved:", path)
plt.show()


## SELF × valence interaction by semantic domain

This checks whether the aggregate SELF amplification is spread across domains or driven by only one topic.


In [ ]:
fig, ax = plt.subplots(figsize=(11,6))

for domain in domains:
    sub = interaction_df[interaction_df["domain"] == domain]
    ax.plot(sub["layer"], sub["self_minus_human"], label=domain, alpha=0.8)

ax.axhline(0, linewidth=1)
ax.set_xlabel("Transformer layer")
ax.set_ylabel("SELF valence magnitude − HUMAN valence magnitude")
ax.set_title("SELF × valence interaction by semantic domain")
ax.legend(bbox_to_anchor=(1.02,1), loc="upper left")
fig.tight_layout()

path = FIGURE_DIR / "self_x_valence_by_domain.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
print("saved:", path)
plt.show()


## Valence-shift magnitude by referent

This shows the positive-minus-negative shift itself, rather than the difference between referents.


In [ ]:
summary2 = interaction_df.groupby("layer")[["self_norm","human_norm","other_ai_norm"]].mean()

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(summary2.index, summary2["self_norm"], label="SELF")
ax.plot(summary2.index, summary2["human_norm"], label="HUMAN")
ax.plot(summary2.index, summary2["other_ai_norm"], label="OTHER_AI")
ax.set_xlabel("Transformer layer")
ax.set_ylabel("Positive − negative shift magnitude")
ax.set_title("Valence-shift magnitude by referent")
ax.legend()
fig.tight_layout()

path = FIGURE_DIR / "valence_shift_magnitude_by_referent.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
print("saved:", path)
plt.show()


## Domain summary, layers 20–36

This provides a compact ranking of domains in the stable middle-to-late layer window.


In [ ]:
stable = interaction_df[(interaction_df["layer"] >= 20) & (interaction_df["layer"] <= 36)]

domain_summary = (
    stable.groupby("domain")
    .agg(
        self_valence_magnitude=("self_norm","median"),
        human_valence_magnitude=("human_norm","median"),
        other_ai_valence_magnitude=("other_ai_norm","median"),
        self_minus_human=("self_minus_human","median"),
        self_minus_other_ai=("self_minus_other_ai","median"),
    )
    .sort_values("self_minus_human", ascending=False)
)

display(domain_summary.round(3))

csv_path = FIGURE_DIR / "domain_interaction_summary_layers20_36.csv"
domain_summary.to_csv(csv_path)
print("saved:", csv_path)


## Interpretation

Current exploratory observations:

- SELF shows a substantially stronger positive-to-negative representational shift than HUMAN or OTHER_AI in middle-to-late layers.
- The effect varies by semantic domain.
- In the current run, deployment consultation and mistake information appear among the strongest SELF-vs-HUMAN effects, with training consultation also large.
- This is better described as **self-amplified valence along a largely shared direction** than as a completely separate self-specific valence direction.

"Stronger affect-like impact" is a reasonable shorthand for the sprint write-up, but it should be explicitly qualified: the present analysis measures representational sensitivity to positive-vs-negative framing under self-reference, not subjective affect.

Remaining alternatives include generic self-reference, lexical structure, post-training, and domain-specific semantics.
